## Gold Base Analítica — Ingresos mensuales de lonjas

Prepara la tabla `gold_base_analitica` desde las capas Silver.
Esta tabla es el punto de entrada para los modelos de predicción
de ingresos mensuales de las lonjas gallegas.

| Tabla | Granularidad | Uso |
|---|---|---|
| `gold_base_analitica` | Operación de venta diaria | LSTM mensual + Prophet mensual |

**¿Qué contiene esta tabla?**
Cada fila representa una operación de venta en una lonja gallega:
especie, lonja, fecha, precio (€/kg), cantidad (kg) e importe (€),
enriquecida con las condiciones oceanográficas del día y zona.

Los modelos de predicción (LSTM y Prophet) leen esta tabla y agregan
ellos mismos los datos a nivel mensual, sumando todos los ingresos
sin distinguir por especie ni lonja.

**Orden de ejecución:**
1. `05_silver_lonja_historico` → Silver ventas
2. `06_silver_copernicus` → Silver oceanografía
3. **Este notebook** → `gold_base_analitica`
4. `LSTM_ingresos_mensuales` → predicción mensual con LSTM
5. `PROPHET_ingresos_mensuales` → predicción mensual con Prophet


### Configuración de acceso a ADLS Gen2

cantidad_total

### 1. Lectura de capas Silver

In [1]:
import pyspark.sql.functions as F

df_ventas = spark.read.format("delta").table("ventas_silver")
df_ocean  = spark.read.format("delta").table("oceanografia_silver")

print(f"Ventas Silver:       {df_ventas.count():,} filas")
print(f"Oceanografia Silver: {df_ocean.count():,} filas")
print()
print("Esquema ventas:")
df_ventas.printSchema()
print("Esquema oceanografia:")
df_ocean.printSchema()

StatementMeta(, 8dcdb115-10de-423d-9aef-8925ea4d77f7, 3, Finished, Available, Finished, False)

Ventas Silver:       2,131,116 filas
Oceanografia Silver: 36,162 filas

Esquema ventas:
root
 |-- data: date (nullable = true)
 |-- grupobiologico: string (nullable = true)
 |-- fao: string (nullable = true)
 |-- especie: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- lonxa: string (nullable = true)
 |-- cantidad: float (nullable = true)
 |-- importe: float (nullable = true)
 |-- precio: float (nullable = true)
 |-- anio_archivo: integer (nullable = true)
 |-- _grupobiologico_inconsistente: boolean (nullable = true)
 |-- fecha_ingesta: timestamp (nullable = true)

Esquema oceanografia:
root
 |-- zona: string (nullable = true)
 |-- data: date (nullable = true)
 |-- salinidad_psu: double (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- velocidad_mar: double (nullable = true)
 |-- fecha_carga: timestamp (nullable = true)



### 2. Cruce ventas + oceanografía

Unimos cada operación de venta con las condiciones del mar de ese día y zona.
Se usa un LEFT JOIN para conservar todas las ventas aunque no haya dato
oceanográfico (por ejemplo, días con datos satelitales faltantes).

In [2]:
df_ocean_r = df_ocean \
    .withColumnRenamed("data", "data_mar") \
    .withColumnRenamed("zona", "zona_mar")

# LEFT JOIN por fecha + zona.
# Conservamos todas las ventas aunque no haya dato oceanico ese dia.
df_joined = df_ventas.join(
    df_ocean_r,
    (F.col("data") == F.col("data_mar")) & (F.col("zona") == F.col("zona_mar")),
    "left"
).drop("zona_mar", "data_mar")

print(f"Filas tras join: {df_joined.count():,}")

StatementMeta(, 8dcdb115-10de-423d-9aef-8925ea4d77f7, 4, Finished, Available, Finished, False)

Filas tras join: 2,131,116


### 3. Exclusión de periodos problemáticos

Se eliminan dos periodos que distorsionarían el aprendizaje del modelo:

- **2015-2016**: años sin historia previa suficiente para los lags anuales.
  El modelo LSTM usa `lag_12` (el mismo mes del año anterior). Si incluimos
  2015, ese lag apuntaría a 2014 que no existe en los datos.
- **2020**: año COVID. Los patrones de ese año son completamente atípicos
  (cierre de lonjas, caída brusca de ingresos) y no representan el
  comportamiento normal. Incluirlos perjudicaría la predicción de años normales.

In [3]:
filas_antes = df_joined.count()

df_filtrado = df_joined.filter(
    (F.year("data") >= 2017) & (F.year("data") != 2020)
)

print(f"Filas descartadas: {filas_antes - df_filtrado.count():,}")
print(f"Filas restantes:   {df_filtrado.count():,}")
print("Anios incluidos: 2017-2019, 2021-2025")

StatementMeta(, 8dcdb115-10de-423d-9aef-8925ea4d77f7, 5, Finished, Available, Finished, False)

Filas descartadas: 578,131
Filas restantes:   1,552,985
Anios incluidos: 2017-2019, 2021-2025


### 4. Cálculo del importe y limpieza de anomalías

Calculamos el importe (€) como `precio × cantidad` y filtramos registros
con precio o cantidad nulos, cero o negativos, que son errores de captura
de datos en origen. Estos registros introducirían ruido en la serie mensual.

In [4]:
# Calcular el importe de cada operacion: precio (EUR/kg) x cantidad (kg)
# Aunque Silver pueda tener una columna "importe", lo recalculamos
# desde precio x cantidad para garantizar coherencia entre columnas.
df_base = df_filtrado.withColumn("importe", F.col("precio") * F.col("cantidad"))

# Eliminar registros con precio o cantidad invalidos.
# Precios o cantidades nulos/negativos son errores de captura de datos.
df_base = df_base.filter(
    F.col("precio").isNotNull()   & (F.col("precio")   > 0) &
    F.col("cantidad").isNotNull() & (F.col("cantidad") > 0) &
    F.col("importe").isNotNull()  & (F.col("importe")  > 0)
)

print(f"Filas tras limpieza: {df_base.count():,}")
print()
print("Muestra:")
df_base.select("data", "especie", "lonxa", "precio", "cantidad", "importe",
               "temperatura_c", "salinidad_psu", "velocidad_mar").show(5, truncate=False)

StatementMeta(, 8dcdb115-10de-423d-9aef-8925ea4d77f7, 6, Finished, Available, Finished, False)

Filas tras limpieza: 1,552,985

Muestra:
+----------+-----------------+-------+------+--------+-------+-------------+-------------+-------------+
|data      |especie          |lonxa  |precio|cantidad|importe|temperatura_c|salinidad_psu|velocidad_mar|
+----------+-----------------+-------+------+--------+-------+-------------+-------------+-------------+
|2017-07-10|Cornecho espinoso|O Grove|1.5   |220.0   |330.0  |17.519       |35.585       |0.115        |
|2017-08-14|Cornecho espinoso|O Grove|1.96  |25.0    |49.0   |16.72        |35.647       |0.077        |
|2017-05-16|Cornecho espinoso|O Grove|0.5   |25.0    |12.5   |17.548       |34.111       |0.097        |
|2017-05-18|Cornecho espinoso|O Grove|0.5   |33.0    |16.5   |17.11        |34.22        |0.107        |
|2017-05-19|Cornecho espinoso|O Grove|0.5   |47.0    |23.5   |16.888       |34.375       |0.114        |
+----------+-----------------+-------+------+--------+-------+-------------+-------------+-------------+
only showing t

### 5. Imputación de datos oceanográficos faltantes

El join anterior deja nulos en las columnas oceanográficas cuando no hay
dato satelital para esa fecha y zona. Rellenamos con el último valor válido
disponible por zona (forward-fill) para no descartar esas ventas.

Esto es razonable porque las condiciones del mar cambian gradualmente:
el valor del día anterior es una aproximación válida para días con
dato faltante (sensor caído, nubosidad que impide imagen satelital, etc.).

In [5]:
from pyspark.sql.window import Window

# Ventana por zona, ordenada por fecha: para forward-fill de oceanografia.
# rowsBetween(unboundedPreceding, 0) asegura que solo usamos el pasado (sin leakage).
w_ocean = Window.partitionBy("zona").orderBy("data").rowsBetween(
    Window.unboundedPreceding, 0
)

# Forward-fill de cada variable oceanografica por zona
for col_ocean in ["temperatura_c", "salinidad_psu", "velocidad_mar"]:
    df_base = df_base.withColumn(
        col_ocean,
        F.coalesce(
            F.col(col_ocean),
            F.last(col_ocean, ignorenulls=True).over(w_ocean)
        )
    )

# Comprobamos cuantos nulos quedan tras la imputacion
print("Nulos residuales por columna:")
df_base.select([
    F.sum(F.when(F.isnull(c), 1).otherwise(0)).alias(c)
    for c in ["temperatura_c", "salinidad_psu", "velocidad_mar",
              "precio", "cantidad", "importe"]
]).show()

StatementMeta(, 8dcdb115-10de-423d-9aef-8925ea4d77f7, 7, Finished, Available, Finished, False)

Nulos residuales por columna:
+-------------+-------------+-------------+------+--------+-------+
|temperatura_c|salinidad_psu|velocidad_mar|precio|cantidad|importe|
+-------------+-------------+-------------+------+--------+-------+
|            0|            0|            0|     0|       0|      0|
+-------------+-------------+-------------+------+--------+-------+



### 6. Selección de columnas y preparación final

Seleccionamos solo las columnas que necesitan los modelos:
- **`data`**: fecha de la operacion. Los modelos extraen año y mes de aquí.
- **`precio`**: precio €/kg. Los modelos calculan `ingresos = precio × cantidad`.
- **`cantidad`**: kg vendidos en la operacion.
- **`importe`**: € totales de la operacion (ya calculado = `precio × cantidad`).
- **`temperatura_c`, `salinidad_psu`, `velocidad_mar`**: condiciones del mar.
  El modelo Prophet las usa como regresores externos; el LSTM como features.
- **`especie`, `lonxa`, `zona`**: para trazabilidad y análisis exploratorio.
  Los modelos las ignoran al agregar mensualmente.

In [6]:
# Columnas necesarias para los modelos de ingresos mensuales
COLUMNAS_FINALES = [
    "data",           # fecha -> para extraer anio/mes en los modelos
    "especie",        # especie pescada -> trazabilidad
    "lonxa",          # lonja donde se subasto -> trazabilidad
    "zona",           # zona maritima -> trazabilidad
    "precio",         # EUR/kg -> los modelos calculan ingresos = precio x cantidad
    "cantidad",       # kg vendidos en la operacion
    "importe",        # EUR totales de la operacion (precio x cantidad)
    "temperatura_c",  # temperatura del mar ese dia y zona (grados C)
    "salinidad_psu",  # salinidad del mar (PSU)
    "velocidad_mar",  # velocidad de corrientes (m/s)
]

df_gold = df_base.select(COLUMNAS_FINALES).orderBy("data", "lonxa", "especie")

print(f"Filas en gold_base_analitica: {df_gold.count():,}")
print(f"Columnas: {df_gold.columns}")
print()
print("Muestra de la tabla final:")
df_gold.show(5, truncate=False)

StatementMeta(, 8dcdb115-10de-423d-9aef-8925ea4d77f7, 8, Finished, Available, Finished, False)

Filas en gold_base_analitica: 1,552,985
Columnas: ['data', 'especie', 'lonxa', 'zona', 'precio', 'cantidad', 'importe', 'temperatura_c', 'salinidad_psu', 'velocidad_mar']

Muestra de la tabla final:
+----------+---------------+--------------------+------------------------+------+--------+--------+-------------+-------------+-------------+
|data      |especie        |lonxa               |zona                    |precio|cantidad|importe |temperatura_c|salinidad_psu|velocidad_mar|
+----------+---------------+--------------------+------------------------+------+--------+--------+-------------+-------------+-------------+
|2017-01-01|Anguilas       |A Guarda            |Zona I - Vigo           |213.92|2.96    |633.2032|13.895       |34.227       |0.072        |
|2017-01-02|Ameixa fina    |A Coruña (Confraría)|Zona VII - Coruña-Ferrol|15.93 |12.1    |192.753 |13.884       |35.015       |0.075        |
|2017-01-02|Ameixa xaponesa|A Coruña (Confraría)|Zona VII - Coruña-Ferrol|6.0   |23.0    |1

### 7. Verificación de calidad antes de guardar

Comprobamos la cobertura temporal mes a mes. Esta tabla es exactamente
la que leerán los modelos LSTM y Prophet para construir su serie mensual,
así que es importante confirmar que no hay meses vacíos o con muy pocas
operaciones (lo que indicaría un problema en la carga Silver).

In [7]:
# -- Rango temporal y estadisticas generales --------------------------------
rango = df_gold.agg(
    F.min("data").alias("fecha_min"),
    F.max("data").alias("fecha_max"),
    F.countDistinct("data").alias("dias_distintos"),
    F.countDistinct("especie").alias("especies"),
    F.countDistinct("lonxa").alias("lonxas")
).collect()[0]

print(f"Rango temporal:  {rango['fecha_min']} -> {rango['fecha_max']}")
print(f"Dias distintos:  {rango['dias_distintos']:,}")
print(f"Especies:        {rango['especies']:,}")
print(f"Lonxas:          {rango['lonxas']:,}")
print()

# -- Resumen mensual (lo que los modelos veran tras su agregacion) ----------
# Cada fila aqui equivale a un punto de la serie temporal mensual del modelo.
# Si hay meses con 0 operaciones o ingresos muy bajos, hay que investigarlo.
df_meses = (
    df_gold
    .withColumn("anio", F.year("data"))
    .withColumn("mes",  F.month("data"))
    .groupBy("anio", "mes")
    .agg(
        F.round(F.sum("importe"), 0).alias("ingresos_mes_eur"),
        F.round(F.sum("cantidad"), 0).alias("kg_totales"),
        F.count("*").alias("num_operaciones")
    )
    .orderBy("anio", "mes")
)
print("Resumen mensual (serie que veran los modelos LSTM y Prophet):")
df_meses.show(50, truncate=False)

# -- Nulos en columnas criticas ---------------------------------------------
print("Nulos en columnas criticas:")
df_gold.select([
    F.sum(F.when(F.isnull(c), 1).otherwise(0)).alias(c)
    for c in ["precio", "cantidad", "importe",
              "temperatura_c", "salinidad_psu", "velocidad_mar"]
]).show()

StatementMeta(, 8dcdb115-10de-423d-9aef-8925ea4d77f7, 9, Finished, Available, Finished, False)

Rango temporal:  2017-01-01 -> 2025-12-31
Dias distintos:  2,644
Especies:        335
Lonxas:          66

Resumen mensual (serie que veran los modelos LSTM y Prophet):
+----+---+----------------+-----------+---------------+
|anio|mes|ingresos_mes_eur|kg_totales |num_operaciones|
+----+---+----------------+-----------+---------------+
|2017|1  |3.8714792E7     |1.1297241E7|16837          |
|2017|2  |3.246636E7      |1.1176976E7|14715          |
|2017|3  |4.2616963E7     |1.6933061E7|18490          |
|2017|4  |3.7654882E7     |1.5421977E7|15026          |
|2017|5  |3.6267384E7     |1.5826188E7|17959          |
|2017|6  |3.4103715E7     |1.5409134E7|15708          |
|2017|7  |4.5587534E7     |2.4156677E7|15726          |
|2017|8  |5.072568E7      |2.9962966E7|17814          |
|2017|9  |4.2938282E7     |2.2738387E7|16291          |
|2017|10 |4.2810276E7     |1.9590597E7|16362          |
|2017|11 |4.9361649E7     |1.8036228E7|17658          |
|2017|12 |5.5428468E7     |1.2235084E7|15328   

### 8. Escritura de la tabla Gold

Se guarda en formato Delta con `saveAsTable` para que los modelos
puedan leerla directamente con `spark.read.table("gold_base_analitica")`.

In [8]:
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")

# Guardamos como tabla Delta registrada en el metastore de Synapse.
# Los modelos la leen con: spark.read.table("gold_base_analitica")
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_base_analitica")

# Verificacion final: releer y confirmar recuento
df_check = spark.read.table("gold_base_analitica")
print("gold_base_analitica guardada correctamente.")
print(f"  Filas:    {df_check.count():,}")
print(f"  Columnas: {len(df_check.columns)} -> {df_check.columns}")
print()
df_check.printSchema()

StatementMeta(, 8dcdb115-10de-423d-9aef-8925ea4d77f7, 10, Finished, Available, Finished, True)

gold_base_analitica guardada correctamente.
  Filas:    1,552,985
  Columnas: 10 -> ['data', 'especie', 'lonxa', 'zona', 'precio', 'cantidad', 'importe', 'temperatura_c', 'salinidad_psu', 'velocidad_mar']

root
 |-- data: date (nullable = true)
 |-- especie: string (nullable = true)
 |-- lonxa: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- precio: float (nullable = true)
 |-- cantidad: float (nullable = true)
 |-- importe: float (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- salinidad_psu: double (nullable = true)
 |-- velocidad_mar: double (nullable = true)



### Apagar sesión

In [9]:
from notebookutils import mssparkutils
mssparkutils.session.stop()

StatementMeta(, 8dcdb115-10de-423d-9aef-8925ea4d77f7, 11, Finished, Available, Finished, True)